# Dannyl MOUHEB – Modèle churn ClientCo – 4. La fréquence de scoring comme hyperparamètre

**Principe** : on teste un pas de scoring de 7, 14, 21 et 28 jours (et en option plusieurs profondeurs d'arbres XGBoost).
- Clients : 60 % apprentissage, 20 % **validation** (pour choisir), 20 % **test** (pour mesurer, jamais utilisés pour choisir).
- Temps : apprentissage du 1er octobre 2023 au 15 janvier 2024 ; listes de relance du 1er février au ~25 mars 2024.
- Capacité commerciale identique pour toutes les fréquences ; un client n'est appelé qu'une fois dans la fenêtre.
- **Critère de choix** : marge annuelle maximale atteignable (groupe) sur la validation, puis la précision.

Durée : quelques minutes (jusqu'à une photo par semaine).

In [ ]:
# --- Mise en place (ne pas modifier) -------------------------------------------------------------
# Le notebook se place à la racine du dépôt (où se trouve data/) et rend importables les modules
# du dossier dannyl_modele_churn.
import os, sys, json
import pandas as pd
from IPython.display import Image, display
if os.path.basename(os.getcwd()) == "dannyl_modele_churn":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("dannyl_modele_churn"))
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
RES = os.path.join("dannyl_modele_churn", "resultats")
print("Dossier de travail :", os.getcwd())

In [ ]:
PAS_JOURS = [7, 14, 21, 28]     # fréquences testées
PROFONDEURS = [5]               # ex. [3, 5, 7] pour optimiser aussi la profondeur des arbres (plus long)
CONTACTS_PAR_MOIS = 20

In [ ]:
import importlib, dannyl_4_frequence_optimisee as script
importlib.reload(script)
sys.argv = ["dannyl_4_frequence_optimisee"] + ["--pas", *map(str, PAS_JOURS), "--profondeurs", *map(str, PROFONDEURS), "--contacts-par-mois", str(CONTACTS_PAR_MOIS)]
script.main()

## Résultats

In [ ]:
O = json.load(open(os.path.join(RES, "frequences", "optimisation_frequence.json"), encoding="utf-8"))
print("Fréquence retenue sur la validation :", O["choix"])
T = pd.read_csv(os.path.join(RES, "frequences", "optimisation_frequence.csv"), sep=";")
display(T.pivot_table(index=["pas_jours", "profondeur"], columns="jeu",
                      values=["marge_max_an_groupe_meur", "precision", "part_perte_captee", "silence_median_au_contact_j"]))
display(Image(os.path.join(RES, "frequences", "optimisation_frequence.png")))